# CareerCompass: Predictive Match Classifier & Class Imbalance Evaluation
**Cohort**: CSE VII, Batch 1 | **Project**: CP-01 CareerCompass | **Date**: September 2026

This notebook documents the feature engineering, model training, and rigorous evaluation of the candidate-job fit classifier.
- **Addressing Class Imbalance**: ~19.5% positive match rate handled with balanced loss weighting.
- **Model Comparison**: Logistic Regression vs. Random Forest vs. HistGradientBoosting.
- **Metric Defense**: Justification of Precision, Recall, and F1-score for recruitment intelligence.


In [1]:
import joblib
import json
import numpy as np
import pandas as pd
import os

base_dir = r'C:\Users\priya\.gemini\antigravity\scratch\careercompass'
metrics_file = os.path.join(base_dir, 'models', 'model_evaluation_metrics.json')
with open(metrics_file, 'r') as f:
    metrics = json.load(f)

print('Loaded Trained Model Evaluation Artifacts.')
print('Best Model Selected:', metrics['best_model'])


Loaded Trained Model Evaluation Artifacts.
Best Model Selected: Random Forest Classifier


### 1. Model Comparison Table across Key Performance Metrics
We evaluated three model architectures on identical 70/15/15 stratified splits:
1. **Logistic Regression (Linear Baseline)**: Fast, interpretable linear decision boundary.
2. **Random Forest Classifier (Non-linear Ensemble)**: Handles non-linear feature interactions and feature importance.
3. **HistGradientBoosting Classifier**: Optimized gradient boosting trees on binned numerical features.


In [1]:
comp_df = pd.DataFrame(metrics['models_evaluated']).T
print(comp_df[['precision', 'recall', 'f1', 'roc_auc', 'pr_auc']])


                                 precision  recall      f1  roc_auc  pr_auc
Logistic Regression (Baseline)      0.7371  0.9207  0.8187   0.9870  0.9597
Random Forest Classifier            1.0000  1.0000  1.0000   1.0000  1.0000
HistGradientBoosting Classifier     1.0000  1.0000  1.0000   1.0000  1.0000


### 2. Feature Importance & Contribution Breakdown
Feature importances demonstrate what drives fit decisions:
- `skill_overlap_ratio` (26.2%) and `semantic_similarity` (24.2%) dominate predictive weight.
- `experience_fit_score` (17.3%) and `experience_delta` (13.1%) enforce tenure alignment.
- `role_discipline_match` (11.2%) ensures cross-domain boundaries are respected.


In [1]:
fi = pd.Series(metrics['feature_importances']).sort_values(ascending=False)
print('Feature Importances in Random Forest Production Model:')
for feat, imp in fi.items():
    print(f'- {feat:25s}: {imp:.4f} ({imp*100:.1f}%)')


Feature Importances in Random Forest Production Model:
- skill_overlap_ratio      : 0.2619 (26.2%)
- semantic_similarity      : 0.2416 (24.2%)
- experience_fit_score     : 0.1734 (17.3%)
- experience_delta         : 0.1314 (13.1%)
- role_discipline_match    : 0.1115 (11.2%)
- missing_skills_count     : 0.0742 (7.4%)
- education_fit_score      : 0.0059 (0.6%)


### 3. Metric Defense: Which Metric Matters Most Here and Why?
> **Assessment Question Response**:
>
> In automated applicant tracking systems, recruiters traditionally prioritize **Precision** because their primary bottleneck is manual screening bandwidth—they cannot afford false positives (unqualified candidates advancing to interviews). However, for a candidate coaching platform like **CareerCompass**, a pure focus on Precision is harmful: it creates false negatives, discouraging viable candidates.
>
> Therefore, the **Balanced F1-Score** (and Area Under the Precision-Recall Curve, **PR-AUC**) is the definitive metric for this application. Because strong matches constitute only ~19.5% of applications, standard Accuracy is deceptive (an all-negative classifier would achieve 80.5% accuracy). F1-score harmonic balancing ensures the model maintains both high selectivity and high candidate discovery.
